# Phase 1.7 — the scaled embedding cache (tile 32UNU, 115 cubes)

**This notebook computes nothing scientific.** It builds one artefact: the
frozen-encoder cache for the 115-cube 32UNU set, so that P1, P2 and P3 can be
re-run at 5.75× the sample size without any of them changing a line of code.

**Why this exists.** Every probe result so far rests on 20 cubes of one tile in
one year, and Phase 1.5b already demonstrated that conclusions from that sample
size do not survive scaling — at 115 cubes the estimator ordering *reversed* and
the first CI-excluding-zero result in the project appeared. P2 inherits the same
limitation in a sharper form: no single-image encoder is separable from the
hand-crafted baseline on gate K2, and the structural hypothesis is undeterminable
because the two encoders it concerns swap rank between fold modes. Both are
sample-size problems. This notebook removes the sample-size excuse.

**Why it needs a GPU and the probes do not.** P4 could scale trivially because it
reads no embeddings — only NDVI and the in-cube weather. P1/P2/P3 read the
Phase 1.2 `.npz` cache, which exists for exactly 20 cubes. Growing that set means
running 1580 retained frames through four real networks. That is the only part
of the whole project that wants a GPU; every probe downstream stays cheap CPU
work.

### What it writes, and where

```
data/scaled_32UNU/raw/            115 cubes   (already there if 1.5b ran)
data/scaled_32UNU/embeddings/     575 .npz    115 cubes x 5 encoders   <- NEW
data/scaled_32UNU/masks/          115 .npz    per-pixel, for common-masking  <- NEW
```

**It does not touch `data/phase1_2/`.** That cache is keyed to exactly 20 cubes
and `assert_embeddings_complete` for the 20-cube manifest must keep passing —
every existing result has to stay reproducible. The scaled cache is a second,
independent, internally-complete directory, and it lives beside the shared cubes
rather than inside a phase folder because several later phases read it.

### The check that makes this trustworthy

The 20 original cubes are a **strict subset** of the 115. Step 10 re-encodes
them into the new cache and compares against the old one: the frame selection,
timestamps and clear fractions must be **bit-identical** (they come from the
cube file, not the network), and the embeddings must agree to a stated
tolerance. If they do not, the scaled cache is not the same experiment and
nothing may be compared across the two.

## Step 1: Install, then restart

In [ ]:
import importlib.util, os, IPython
SENTINEL = "/content/.phase1_7_installed"
try:
    import google.colab            # noqa: F401
    ON_COLAB = True
except ImportError:
    # find_spec("google.colab") is NOT equivalent: it raises rather than
    # returning None when the parent `google` package is absent.
    ON_COLAB = False

if not ON_COLAB:
    print("not on Colab: skipping the install and the restart.")
    print("Run against your own environment (pip install -r requirements.txt) "
          "and continue from Step 2.")
elif os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    # satlaspretrain-models is REQUIRED here and was not in Phase 1.5's list:
    # that phase read no embeddings, this one builds them.
    !pip install earthnet s3fs xarray zarr netCDF4 scikit-learn scipy satlaspretrain-models

    # Colab ships a CUDA-matched torch. Installing over it swaps in a CPU wheel
    # and makes every encoder far slower, so only act if something is MISSING.
    # This is the one notebook in the project where that distinction costs real
    # wall-clock: everything downstream is CPU-only by design.
    if (importlib.util.find_spec("torch") is None
            or importlib.util.find_spec("torchvision") is None):
        !pip install torch torchvision

    # Verify before restarting, so a broken install cannot reach the encoders.
    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, pandas, numpy, "
             "torch, torchvision, satlaspretrain_models, sklearn, scipy")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the encoders."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/paths.py",
            "data/climatology.py", "encoders/manifest.py",
            "encoders/pipeline.py", "encoders/base.py", "encoders/frames.py",
            "encoders/raw_features.py", "encoders/imagenet_vit.py",
            "encoders/dinov2_vit.py", "encoders/satlas_s2.py",
            "encoders/satlas_s2_mi.py", "scripts/scale_p4.py",
            "probes/cv.py", "probes/p1_appearance.py", "probes/p2_deltas.py",
            "tests/test_cv_folds.py", "tests/test_p2_deltas.py",
            "tests/conftest.py"]
ZIP_NAME = "phase1_7_repo.zip"
PHASE = "phase1_7"
INPUT_PHASE = "phase1_2"          # resolved by the shared block; NEVER read here

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for d in ("*", "*/*", "*/*/*")
                for h in glob.glob(f"{DRIVE}/{d}/probes/cv.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "probes", "p4_ceiling.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
            print()
            print("=" * 70)
            print("THE NOTEBOOK FILE ON DISK WAS JUST REPLACED.")
            print("Colab is still showing the cells it opened. To pick up the")
            print("new ones: File > Open notebook > Google Drive, and open")
            print("   " + os.path.join(REPO, "notebooks"))
            print("Until you do, the .py files are new and these cells are old.")
            print("=" * 70)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    # Off Colab, walk up from the working directory: running the notebook from
    # notebooks/ is normal and must not be mistaken for a missing checkout.
    d = os.getcwd()
    while not looks_like_repo(d) and os.path.dirname(d) != d:
        d = os.path.dirname(d)
    REPO = d

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.7 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Make a NEW subfolder  My Drive / NeurIPS-CCAI-2026 / phase1_7
          4. Drag {ZIP_NAME} into it (do not unzip)
          5. Re-run this cell.

        One subfolder per phase is deliberate: deleting phase1_7/ removes
        everything Phase 1.7 created and nothing an earlier phase depends on.
        data/raw stays at the project root -- it is shared, not a phase.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

from data.paths import RAW_DIR, describe_phase, phase_dir

# --- READ-ONLY inputs, resolved wherever they already live -----------------
# === RESOLVER (pinned by tests/test_notebook_resolver.py) -- BEGIN ===
# Extracted and exercised by that test against a simulated Drive tree, so
# the precedence rule below cannot silently regress into "first hit wins".
def _candidates(rel, pattern="*"):
    """Every directory on Drive that could be `rel`, with its file count.

    Searched: this checkout, then Drive one, two and three levels down. Three,
    because phases are subfolders of one project folder -- the Phase 1.2
    embeddings sit at
        MyDrive / NeurIPS-CCAI-2026 / phase1_2 / data/phase1_2/embeddings
    which is two wildcards, while the shared cubes at
        MyDrive / NeurIPS-CCAI-2026 / data/raw
    are one.
    """
    seen, out = set(), []
    cands = [os.path.join(REPO, rel)]
    if DRIVE:
        for depth in ("*", "*/*", "*/*/*"):
            cands += sorted(glob.glob(f"{DRIVE}/{depth}/{rel}"))
    for c in cands:
        c = os.path.abspath(c)
        if c in seen or not os.path.isdir(c):
            continue
        seen.add(c)
        out.append((c, len(glob.glob(os.path.join(c, pattern)))))
    return out


def _resolve(rel, pattern, label, foreign_phase=False):
    """Pick ONE directory, by evidence, and show every candidate considered.

    TAKING THE FIRST HIT IS NOT A SELECTION, and it cost a real run: a stale
    copy of data/phase1_2/embeddings sat INSIDE the phase1_3 checkout, the old
    "this checkout first" rule preferred it over the true Phase 1.2 folder, and
    the run died on a pre-schema file nobody knew was there.

    So: most files wins, and for ANOTHER phase's artefacts a directory inside
    THIS phase's checkout never beats one outside it, whatever the counts. That
    is the layout contract -- a phase reads its inputs in place and never owns
    a copy -- expressed as code rather than as a docstring.
    """
    cands = [(c, n) for c, n in _candidates(rel, pattern) if n > 0]
    if not cands:
        return os.path.join(REPO, rel), []        # the caller reports the gap
    repo_abs = os.path.abspath(REPO)

    def inside_repo(c):
        return os.path.commonpath([repo_abs, c]) == repo_abs

    # The penalty applies ONLY in a per-phase checkout. In a plain development
    # clone the repo root IS where data/phase1_2 belongs, so penalising "inside
    # the repo" there would be backwards -- and a warning that fires when
    # nothing is wrong is a warning nobody reads the second time.
    def demote(c):
        return foreign_phase and IS_PHASE_CHECKOUT and inside_repo(c)

    ranked = sorted(cands, key=lambda cn: (
        0 if demote(cn[0]) else -1,                           # outside first
        -cn[1],                                               # then the fullest
        len(cn[0]),                                           # then the shortest
    ))
    chosen = ranked[0][0]
    if len(cands) > 1:
        print(f"[resolve] {label}: {len(cands)} candidate directories hold files --")
        for c, n in ranked:
            mark = "  <- USING" if c == chosen else ""
            flag = "  [inside this checkout]" if inside_repo(c) else ""
            print(f"[resolve]     {n:>4} file(s)  {c}{flag}{mark}")
    if demote(chosen):
        print(f"[resolve] WARNING: {label} resolved INSIDE this phase's checkout:")
        print(f"[resolve]   {chosen}")
        print("[resolve] Another phase's artefacts do not belong here -- one phase")
        print("[resolve] reads another's in place and never owns a copy. This is")
        print("[resolve] almost certainly stale. Delete it and re-run Step 2 so")
        print("[resolve] the real directory is found.")
    return chosen, ranked


# Is this checkout a PHASE folder (Drive), or a plain clone (local dev)? The
# name settles it and covers both Drive layouts that have existed: the nested
# "NeurIPS-CCAI-2026/phase1_3" and the older sibling "…-2026-phase1_3".
IS_PHASE_CHECKOUT = PHASE in os.path.basename(os.path.abspath(REPO))

RAW, _raw_cands = _resolve(RAW_DIR, "*.nc", "RAW")
EMB_IN, _emb_cands = _resolve(os.path.join("data", INPUT_PHASE, "embeddings"),
                              "*.npz", "EMB_IN", foreign_phase=True)
os.makedirs(RAW, exist_ok=True)

# A phase checkout should not contain another phase's artefact tree at all,
# even an empty one: it shadows the real directory on every future run.
_intruder = os.path.join(REPO, "data", INPUT_PHASE)
if IS_PHASE_CHECKOUT and os.path.isdir(_intruder):
    print()
    print(f"[resolve] NOTE: {_intruder}")
    print(f"[resolve] exists inside the {PHASE} checkout. {INPUT_PHASE} "
          "artefacts belong in the")
    print(f"[resolve] {INPUT_PHASE} subfolder. Nothing here writes to it, but it "
          "will keep shadowing")
    print("[resolve] the real one until you delete it.")
# === RESOLVER -- END ===

# --- this phase's OWN outputs ----------------------------------------------
RESULTS = phase_dir(PHASE, "results")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
n_emb = len(glob.glob(os.path.join(EMB_IN, "*.npz")))
print(f"\nREPO    {REPO}")
print(f"RAW     {RAW}   ({n_cubes} cubes)"
      + ("" if n_cubes else "   <- Step 4 downloads them"))
print(f"EMB_IN  {EMB_IN}   ({n_emb} .npz)")
print("        ^ the 20-cube Phase 1.2 cache. READ-ONLY here, and NEVER written")
print("          to: it is keyed to exactly 20 cubes and every published result")
print("          must stay reproducible from it. Step 10 reads it to prove the")
print("          scaled cache reproduces it on the cubes the two share.")
print(f"RESULTS {RESULTS}   (this phase writes here only)")
describe_phase(PHASE)

from data.ndvi import ndvi
from encoders import TIER_A
from encoders.manifest import build_manifest
from probes import cv
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, "
      f"splits at {cv.__name__}, modes {cv.MODES}")
print(f"encoders to cache ({len(TIER_A)}): {TIER_A}")
print("this notebook writes a CACHE, not a result. No probe runs here.")
for f in REQUIRED:
    print(f"  ok  {f}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
import shlex, subprocess

PY = shlex.quote(sys.executable)

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Environment check — **and this is the one phase that wants a GPU**

Encoding is the only GPU-bound work in the project. It will run on CPU (Phase
1.2 measured DINOv2 at 42 s for 20 cubes there), just slower. Everything
downstream of this notebook is CPU-only by design.

In [ ]:
import glob, os, textwrap

import numpy as np, pandas as pd, sklearn, scipy
print(f"numpy {np.__version__} | pandas {pd.__version__} | "
      f"sklearn {sklearn.__version__} | scipy {scipy.__version__}")

import torch
print(f"torch {torch.__version__}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    DEVICE = "cuda"
    print(f"GPU  {p.name}, {p.total_memory / 1e9:.1f} GB  <- encoding will use this")
else:
    DEVICE = "cpu"
    print(textwrap.dedent("""
        No GPU. This still WORKS -- nothing here needs CUDA -- but 1580 frames
        through four networks will take a while. On Colab: Runtime > Change
        runtime type > T4 GPU, then re-run from Step 2.
    """).strip())
print(f"\nDEVICE = {DEVICE!r}")

# Python >= 3.10, and this one is not cosmetic. dinov2_vitb14 loads its code
# from torch.hub, and that code uses PEP 604 unions (`X | None`), so on 3.9 it
# dies with "unsupported operand type(s) for |: 'type' and 'NoneType'" -- a
# TypeError from inside a downloaded file, five steps after the real cause.
# Measured on this project's own 3.9 venv: the other four encoders build fine
# and only DINOv2 fails, which is exactly the kind of partial failure that
# produces a cache with a hole in it.
import sys
_py = sys.version_info
print(f"python {_py.major}.{_py.minor}.{_py.micro}")
assert _py >= (3, 10), (
    f"python {_py.major}.{_py.minor} is too old for dinov2_vitb14: its "
    "torch.hub code uses `X | None`, which is a syntax error before 3.10. The "
    "other four encoders would build and the cache would silently be missing "
    "one encoder for every cube. Colab is on 3.11+; if you are running this "
    "locally, use a 3.10+ interpreter."
)
print("\nNOTHING here is fine-tuned. Every encoder is frozen: .eval() and")
print("torch.no_grad(), re-asserted on every call by encoders.base.FrozenEncoder.")

## Step 4: The 115 cubes

Downloaded by the same function Phase 1.5b used, so this set is *the same set*
P4's ceiling was measured on — the scaled P2 numbers will sit beside the scaled
P4 numbers over identical weather realisations. Idempotent: it skips whatever is
already on disk.

The cubes live beside `data/raw` rather than inside this phase's folder, because
`data/raw` is shared and so is this: deleting the phase folder must not delete
363 MB of cubes.

In [ ]:
TILE, N_CUBES = "32UNU", 115
_scaled_rel = os.path.join("data", f"scaled_{TILE}")

# Where the shared, non-phase artefacts live: beside data/raw, one level above a
# per-phase checkout. In a plain development clone that is the repo root.
PROJECT_ROOT = os.path.dirname(REPO) if IS_PHASE_CHECKOUT else REPO

CUBES, _cube_cands = _resolve(os.path.join(_scaled_rel, "raw"), "*.nc", "CUBES")
if not glob.glob(os.path.join(CUBES, "*.nc")):
    CUBES = os.path.join(PROJECT_ROOT, _scaled_rel, "raw")
    print(f"[scaled] nothing on disk yet; will download into {CUBES}")

SCALED_ROOT = os.path.dirname(CUBES)
OUT_EMB = os.path.join(SCALED_ROOT, "embeddings")
OUT_MSK = os.path.join(SCALED_ROOT, "masks")
os.makedirs(OUT_EMB, exist_ok=True)
os.makedirs(OUT_MSK, exist_ok=True)

from scripts.scale_p4 import download
CUBE_PATHS = download(TILE, "train", N_CUBES, CUBES)

n = len(CUBE_PATHS)
print(f"\nCUBES    {CUBES}   ({n} cubes)")
print(f"OUT_EMB  {OUT_EMB}   (this notebook writes here)")
print(f"OUT_MSK  {OUT_MSK}   (this notebook writes here)")
print(f"EMB_IN   {EMB_IN}   ({len(glob.glob(os.path.join(EMB_IN, '*.npz')))} .npz, "
      "the 20-cube Phase 1.2 cache -- READ-ONLY, for the Step 10 cross-check)")
assert n >= 20, f"only {n} cubes; the 64 px non-overlap rule caps 32UNU at 115"

# The 20 original cubes must be a strict subset, or Step 10 has nothing to check
# and no scaled result is comparable to a published one.
OLD_CUBES = {os.path.basename(p) for p in glob.glob(os.path.join(RAW, "*.nc"))}
NEW_CUBES = {os.path.basename(p) for p in CUBE_PATHS}
SHARED = sorted(OLD_CUBES & NEW_CUBES)
print(f"\n{len(OLD_CUBES)} original cubes, {len(NEW_CUBES)} scaled, "
      f"{len(SHARED)} shared, {len(NEW_CUBES - OLD_CUBES)} new")
assert OLD_CUBES <= NEW_CUBES, (
    f"{sorted(OLD_CUBES - NEW_CUBES)[:3]} are in data/raw but NOT in the scaled "
    "set. The 20-cube results would then not be a subset of the scaled ones and "
    "the two could not be compared.")

## Step 5: Unit tests

The invariant is **0 failed**, not a particular pass count.

In [ ]:
# pytest.ini already sets addopts = -q. Passing -q again makes it -qq,
# which hides the per-file progress.
sh(f"{PY} -m pytest tests")

## Step 6: Build the five encoders

This is where weights are downloaded (first run only) and moved to the device.
`build_encoder` asserts the wrapper's own name matches the registry key, so a
mis-wired encoder cannot be silently encoded under another one's filename.

In [ ]:
import time
from encoders import TIER_A, build_encoder

print(f"TIER_A = {TIER_A}\n")
t0 = time.time()
ENCODERS = {}
for name in TIER_A:
    ENCODERS[name] = build_encoder(name, device=DEVICE, verbose=True)
print(f"\n{len(ENCODERS)} encoders ready on {DEVICE} in {time.time() - t0:.0f}s")

for name, enc in ENCODERS.items():
    print(f"  {name:<24} D={enc.embed_dim:<5} grid_D={enc.grid_dim:<5} "
          f"window_len={enc.window_len}")
assert set(ENCODERS) == set(TIER_A)

## Step 7: Smoke test on ONE cube before committing to 575 files

A shape or dtype error found here costs seconds; found in Step 8 it costs the
whole run. Also asserts the five encoders agree on *which* frames were retained
— they must, because frame selection happens before any network sees anything.

In [ ]:
from data.loader import load_cube
from encoders.pipeline import encode_cube

probe = load_cube(sorted(CUBE_PATHS)[0], verbose=False)
print(f"smoke cube: {os.path.basename(probe.path)}  values {probe.values.shape}\n")

smoke, t0 = {}, time.time()
for name, enc in ENCODERS.items():
    ec = encode_cube(probe, enc, verbose=False)
    smoke[name] = ec
    assert ec.embeddings.ndim == 2 and ec.embeddings.shape[1] == enc.embed_dim
    assert ec.grid.shape[1:] == (16, enc.grid_dim), ec.grid.shape
    assert np.isfinite(ec.embeddings).all() and np.isfinite(ec.grid).all()
    print(f"  {name:<24} pooled {str(ec.embeddings.shape):<14} "
          f"grid {str(ec.grid.shape):<18} wsd max {ec.window_span_days.max():.0f} d")

ts0 = smoke["raw_features"].timestamps
assert all(np.array_equal(ec.timestamps, ts0) for ec in smoke.values()), (
    "the encoders disagree about which frames were retained; frame selection "
    "happens BEFORE any network sees the data, so they cannot legitimately differ")
print(f"\nall five agree on {len(ts0)} retained frames | {time.time() - t0:.0f}s "
      f"for 1 cube x 5 encoders on {DEVICE}")
print(f"projected for {len(CUBE_PATHS)} cubes: "
      f"~{(time.time() - t0) * len(CUBE_PATHS) / 60:.0f} min")

## Step 8: The encode — resumable, because Colab sessions die

575 `.npz` files. The loop **skips whatever already exists** and re-validates it
on load rather than trusting the filename, so a disconnected session is resumed
by simply re-running this cell — nothing is recomputed and nothing is
half-written (`save_encoded` writes then the loader re-asserts).

A cube that fails is recorded and the run continues; the completeness assertion
in Step 9 is what refuses an incomplete cache. Failing on cube 3 of 115 and
losing the other 112 would be the wrong trade.

In [ ]:
from encoders.pipeline import (SCHEMA_VERSION, cube_masks, encode_cube,
                               load_encoded, save_encoded, save_masks)

print(f"cache schema v{SCHEMA_VERSION}. An older-schema file is REFUSED on load, "
      "never silently reused.")
print(f"writing to {OUT_EMB}\n")

rows, failures = [], []
t_start = time.time()
for i, path in enumerate(sorted(CUBE_PATHS), 1):
    cube = os.path.basename(path)
    stem = os.path.splitext(cube)[0]
    todo = [n for n in TIER_A
            if not os.path.exists(os.path.join(OUT_EMB, f"{stem}__{n}.npz"))]
    mask_path = os.path.join(OUT_MSK, f"{stem}__masks.npz")
    if not todo and os.path.exists(mask_path):
        rows += [{"cube": cube, "encoder": n, "status": "cached"} for n in TIER_A]
        continue

    try:
        s = load_cube(path, verbose=False)
    except Exception as e:                      # noqa: BLE001 -- reported, not hidden
        failures.append((cube, "load_cube", f"{type(e).__name__}: {e}"))
        print(f"[{i:>3}/{len(CUBE_PATHS)}] FAILED to load {cube}: {e}")
        continue

    if not os.path.exists(mask_path):
        save_masks(OUT_MSK, cube_masks(s, verbose=False), verbose=False)

    t0 = time.time()
    for name in TIER_A:
        out = os.path.join(OUT_EMB, f"{stem}__{name}.npz")
        try:
            if os.path.exists(out):
                ec, status = load_encoded(out), "cached"
            else:
                ec = encode_cube(s, ENCODERS[name], verbose=False)
                save_encoded(OUT_EMB, ec, verbose=False)
                status = "encoded"
            rows.append({"cube": cube, "encoder": name, "status": status,
                         "T_kept": int(ec.embeddings.shape[0]),
                         "D": int(ec.embeddings.shape[1])})
        except Exception as e:                  # noqa: BLE001
            failures.append((cube, name, f"{type(e).__name__}: {e}"))
            print(f"[{i:>3}/{len(CUBE_PATHS)}] FAILED {name} on {cube}: {e}")

    done = sum(1 for r in rows if r["status"] == "encoded")
    el = time.time() - t_start
    print(f"[{i:>3}/{len(CUBE_PATHS)}] {cube[:52]:<52} "
          f"{time.time() - t0:5.1f}s | {done:>4} encoded | "
          f"elapsed {el / 60:5.1f} min | eta {el / i * (len(CUBE_PATHS) - i) / 60:5.1f} min",
          flush=True)

ENCODE_LOG = pd.DataFrame(rows)
print(f"\n{len(ENCODE_LOG)} (cube, encoder) pairs in {(time.time() - t_start) / 60:.1f} min")
print(ENCODE_LOG.status.value_counts().to_string())
if failures:
    print(f"\n{len(failures)} FAILURES:")
    for f in failures[:20]:
        print("  ", f)

## Step 9: Audit the cache — all 115 × 5, or it is not usable

A cube silently missing one encoder turns every per-encoder comparison into a
comparison over *different cubes*, and no downstream assertion can detect that.
This is the same audit Phase 1.4 runs, pointed at the new directory.

In [ ]:
from encoders.pipeline import (assert_embeddings_complete, audit_embeddings,
                               print_embedding_audit)

CUBE_IDS = {os.path.basename(p) for p in CUBE_PATHS}
AUDIT = audit_embeddings(OUT_EMB, cube_ids=CUBE_IDS)
print()
assert_embeddings_complete(AUDIT, CUBE_IDS, TIER_A)

n_emb = len(glob.glob(os.path.join(OUT_EMB, "*.npz")))
n_msk = len(glob.glob(os.path.join(OUT_MSK, "*.npz")))
print(f"\nembeddings {n_emb} .npz  (expected {len(CUBE_IDS)} x {len(TIER_A)} "
      f"= {len(CUBE_IDS) * len(TIER_A)})")
print(f"masks      {n_msk} .npz  (expected {len(CUBE_IDS)})")
assert n_emb == len(CUBE_IDS) * len(TIER_A), "the embedding cache has holes"
assert n_msk == len(CUBE_IDS), "the mask cache has holes -- common-masking needs it"
mb = sum(os.path.getsize(p) for p in glob.glob(os.path.join(OUT_EMB, "*.npz")))
print(f"cache size {mb / 1e6:.0f} MB")

## Step 10: The cross-check — the 20 shared cubes must reproduce

The 20 original cubes are a strict subset of the 115, so re-encoding them is a
**direct reproducibility test against a published cache**.

Two different standards, deliberately:

- **Frame selection, timestamps, clear fractions: BIT-IDENTICAL.** These come
  from the cube file and the mask rule, not from a network. Any difference means
  the two caches describe different frames and nothing may be compared.
- **Embeddings: agree to a tolerance, and the number is reported.** These are
  float32 network outputs, and the old cache was written on different hardware
  (CPU/T4) than this run. Phase 1.4 already measured the cost of that: a Colab
  reproduction of P1 moved balanced accuracies by ±0.003. Demanding bit-equality
  here would fail for a reason that has nothing to do with correctness — so the
  difference is measured and printed rather than assumed away.

In [ ]:
from encoders.pipeline import load_encoded

TOL = 1e-3          # stated, not silent: see the markdown above
rows = []
for cube in SHARED:
    stem = os.path.splitext(cube)[0]
    for name in TIER_A:
        old_p = os.path.join(EMB_IN, f"{stem}__{name}.npz")
        new_p = os.path.join(OUT_EMB, f"{stem}__{name}.npz")
        if not os.path.exists(old_p):
            continue
        old, new = load_encoded(old_p), load_encoded(new_p)

        # Bit-identical: these do not come from a network.
        np.testing.assert_array_equal(
            old.kept_idx, new.kept_idx,
            err_msg=f"{cube} x {name}: frame SELECTION differs between the "
                    "20-cube cache and the scaled one. The two describe "
                    "different frames; no result is comparable across them.")
        np.testing.assert_array_equal(old.timestamps, new.timestamps)
        np.testing.assert_allclose(old.clear_frac, new.clear_frac, rtol=0, atol=0)

        d_pool = float(np.abs(old.embeddings - new.embeddings).max())
        d_grid = float(np.abs(old.grid.astype(np.float64)
                              - new.grid.astype(np.float64)).max())
        rows.append({"encoder": name, "cube": cube,
                     "max_abs_pooled": d_pool, "max_abs_grid": d_grid})

CHECK = pd.DataFrame(rows)
assert len(CHECK), "no shared (cube, encoder) pair was compared -- the check is vacuous"
summary = (CHECK.groupby("encoder")
           .agg(n=("cube", "nunique"),
                pooled_max=("max_abs_pooled", "max"),
                grid_max=("max_abs_grid", "max")))
print(f"{len(CHECK)} shared (cube, encoder) pairs re-encoded and compared\n")
print(summary.to_string())
print(f"\nframe selection / timestamps / clear_frac: BIT-IDENTICAL on all "
      f"{len(CHECK)} pairs")
worst = float(CHECK.max_abs_pooled.max())
print(f"largest pooled-embedding difference: {worst:.3g} (tolerance {TOL})")
assert worst <= TOL, (
    f"the scaled cache differs from the 20-cube cache by {worst:.3g}, above the "
    f"{TOL} tolerance. That is not device jitter -- something about the encoding "
    "changed, and the published 20-cube results are not comparable to anything "
    "computed from this cache.")
print("\nThe scaled cache REPRODUCES the published one on the shared cubes.")

## Step 11: What the scaled set actually contains

Printed before any probe touches it, so the sample size every later claim rests
on is on the record. Effective n is **cubes**, and this is the number that
changes.

In [ ]:
from encoders.manifest import assert_strata_present, assert_weather_join, build_manifest

SAMPLES = [load_cube(p, verbose=False) for p in sorted(CUBE_PATHS)]
MANIFEST = build_manifest(SAMPLES)
assert_strata_present(MANIFEST)
print()
JOIN = assert_weather_join(MANIFEST, CUBES)
assert max(JOIN["max_abs_diff"].values()) == 0.0

print(f"\nMANIFEST {MANIFEST.shape} | {MANIFEST.cube_id.nunique()} cubes | "
      f"tiles {sorted(MANIFEST.tile.unique())} | years {sorted(MANIFEST.year.unique())}")
print(f"retained frames {len(MANIFEST)} (was 264 at 20 cubes -- "
      f"{len(MANIFEST) / 264:.1f}x)")
print(f"grid cells      {len(MANIFEST) * 16}")

# The delta pairs P2 will build, and the gap axis it must use.
from probes import p2_deltas as p2
PAIRS = p2.pair_index(MANIFEST, verbose=True)
AXES = p2.assert_gap_axes_disagree(PAIRS, verbose=True)
print(f"\ndelta pairs {PAIRS.n_pairs} (was 244 at 20 cubes -- "
      f"{PAIRS.n_pairs / 244:.1f}x)")
print(f"effective n {MANIFEST.cube_id.nunique()} CUBES (was 20 -- "
      f"{MANIFEST.cube_id.nunique() / 20:.2f}x)")
print("\nThis is the number every interval in the scaled re-run is clustered on.")

## Step 12: Report, and what to run next

The cache is the deliverable. Nothing scientific was computed here, and nothing
should be quoted from this notebook.

In [ ]:
RESULTS = phase_dir(PHASE, "results")
report = os.path.join(RESULTS, "phase1_7_scaled_cache.csv")
ENCODE_LOG.to_csv(report, index=False)
CHECK.to_csv(os.path.join(RESULTS, "phase1_7_reproduction_check.csv"), index=False)
print(f"wrote {report}")
print(f"      {os.path.join(RESULTS, 'phase1_7_reproduction_check.csv')}")
print()
describe_phase(PHASE)

print(textwrap.dedent(f"""
    ------------------------------------------------------------------
    THE SCALED CACHE IS READY.

      cubes       {CUBES}
      embeddings  {OUT_EMB}
      masks       {OUT_MSK}

    P1 and P2 need NO code change to use it -- both already take emb_dir and
    mask_dir as parameters. Re-run them with:

        RESULTS_DF = p2.run_p2(MANIFEST, CUBES,
                               emb_dir={OUT_EMB!r},
                               mask_dir={OUT_MSK!r})

    where MANIFEST is the 115-cube manifest built in Step 11.

    What to expect, from Phase 1.5b's precedent at this exact sample size:
      * K2 separability is a sample-size problem and will probably resolve.
      * The structural-hypothesis rank flip may or may not settle; if the
        ordering is still unstable at 115 cubes, that is a real finding about
        the benchmark and not a missing experiment.
      * The MAGNITUDE result is NOT promised to move. The gap-length control
        beats every encoder outright (+0.209 vs +0.174), and more data tightens
        both estimates rather than reordering them. Report whatever it does.
    ------------------------------------------------------------------
""").strip())

## Phase 1.7 is done when

- [ ] Step 5 reports **0 failed**.
- [ ] Step 8 reports **0 failures** and 575 `(cube, encoder)` pairs.
- [ ] Step 9's audit passes: 575 embedding `.npz` and 115 mask `.npz`, no holes.
- [ ] **Step 10 passes** — frame selection bit-identical on the 20 shared cubes,
      embeddings within tolerance. Without this the scaled cache is a different
      experiment and cannot be compared to any published number.
- [ ] Step 11 prints the manifest, the pair count, and the effective n in cubes.
- [ ] The `embeddings/` and `masks/` directories are on **Drive**, not in the
      Colab container, or the next session loses them.
- [ ] Verbatim stdout archived to `notebooks/runs/`.

**Nothing in this notebook is a result.** It is an input to the scaled re-runs
of P1, P2 and P3.